#### Data extraction
1. Loading SrpWN excel file
2. Extracting serban and english sheets in different datasets
3. Comparing the sysnet IDs, serching for untraslated ones and creating database of untraslated synsets - candidates for adding to SrpwN

In [23]:
#Import libs
import pandas as pd
import numpy as np

In [24]:
#Load srpwn excel file
srpwn_med_filename = "./data/srpwn - medicina.xlsx"

xls = pd.ExcelFile(srpwn_med_filename)

srpwn = pd.read_excel(xls, sheet_name="srpski")
wordnet = pd.read_excel(xls, sheet_name="engleski")

In [25]:
srpwn.describe()

,Unnamed: 0,BCS,ID,literals,def,usage,hypernymID,hypernym_literals,hypernym_def,hypernym_usage
count,497,78,497,497,497,497,464,497,497,497
unique,1,4,478,474,478,52,205,203,204,13
top,medicine,2,ENG30-10164492-n,glavna sestra,Žena koja je zadužena za negu i lečenje u medi...,-,ENG30-14070360-n,-,-,-
freq,497,24,2,2,2,445,34,36,35,455


In [26]:
wordnet.describe()

,domain,BCS,ID,literals,def,usage,hypernymID,hypernym_literals,hypernym_def,hypernym_usage,Prevod literala,Prevod def
count,2383,81,2383,2383,2383,2383,2093,2383,2383,2383,406,406
unique,1,3,2287,2196,2283,319,698,691,698,138,383,387
top,medicine,%,ENG30-14151884-n,-,a rare inherited disorder of fat metabolism; c...,-,ENG30-14336539-n,-,-,-,mongolizam; Daunov sindrom; trizomija 21,a congenital disorder caused by having an extr...
freq,2383,41,3,69,3,2062,82,298,290,2030,3,3


In [27]:
#Number of synsets in SrpWN 
print("Number of synsets in SrpWN:", len(srpwn))
print(srpwn.shape)

#Number of synsets in WordNet
print("Number of synsets in WordNet:", len(wordnet))
print(wordnet.shape)

Number of synsets in SrpWN: 497
(497, 10)
Number of synsets in WordNet: 2383
(2383, 12)


In [28]:
len(wordnet.ID.unique())

2287

In [29]:
len(srpwn.ID.unique())

478

In [30]:
sum(srpwn.ID.value_counts()>1) 

19

In [31]:
sum(wordnet.ID.value_counts()>1)

91

Searching for candidates

In [32]:
#Only in SrpWN
srpwn_only = pd.merge(srpwn, wordnet, on='ID', how='left_anti')
print("Only in SrpWN =", len(srpwn_only))

#Only in WordNet - number of candidates for adding to srpwn
# srpwn_candidates = pd.merge(wordnet, srpwn, on='ID', how='left_anti')
srpwn_candidates = wordnet[~wordnet["ID"].isin(srpwn["ID"])]

print("Number of untranslated records:", len(srpwn_candidates))

Only in SrpWN = 96
Number of untranslated records: 1977


In [33]:
#Detection of missing vales
#It is noted that some columns have symbol "-" as empty value those values as Nan

srpwn.replace("-", np.nan, inplace=True)
print(srpwn.isna().sum())

srpwn.columns[pd.isna(srpwn).any()].tolist()

Unnamed: 0             0
BCS                  419
ID                     0
literals               1
def                    0
usage                445
hypernymID            33
hypernym_literals     36
hypernym_def          35
hypernym_usage       455
dtype: int64


['BCS',
 'literals',
 'usage',
 'hypernymID',
 'hypernym_literals',
 'hypernym_def',
 'hypernym_usage']

In [42]:
#Detection of missing vales
#It is noted that some columns have symbol "-" as empty value those values as Nan

srpwn_candidates.replace("-", np.nan, inplace=True)
print(srpwn_candidates.isna().sum())

srpwn_candidates.columns[pd.isna(srpwn_candidates).any()].tolist()

domain                  0
BCS                  1974
ID                      0
literals               58
def                     0
usage                1748
hypernymID            260
hypernym_literals     267
hypernym_def          260
hypernym_usage       1705
Prevod literala      1977
Prevod def           1977
dtype: int64


['BCS',
 'literals',
 'usage',
 'hypernymID',
 'hypernym_literals',
 'hypernym_def',
 'hypernym_usage',
 'Prevod literala',
 'Prevod def']

In [34]:
def empty(value):
    """Returns True if the cell is empty, NaN or whitespace."""
    if pd.isna(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False
 

In [35]:
srpwn_ids = set(srpwn.ID)
wordnet_ids = set(wordnet.ID)

matched_ids = wordnet_ids & srpwn_ids
unmatched_ids  = wordnet_ids - srpwn_ids

print("Synsets missing in srpwn:", len(unmatched_ids))
print("Sysnets matched in both sheets:", len(matched_ids))
 

Synsets missing in srpwn: 1900
Sysnets matched in both sheets: 387


In [36]:
# print(wordnet[wordnet.ID.duplicated()==True].ID)
duplicates = wordnet.ID.value_counts()
print("Duplicate synsets in WordNet:", (duplicates>1).sum())

print("Unique synsets in WordNet:" , len(wordnet.ID.unique()))
print("In WordNet:" , len(wordnet))

Duplicate synsets in WordNet: 91
Unique synsets in WordNet: 2287
In WordNet: 2383


#### Cuvanje kandidata u poseban excel fajl na kom se nastavlja pilot test modela i razlicitih prompt strategija

In [37]:
# --- Snimi detaljne rezultate za dalju upotrebu ---
pilot = srpwn_candidates.sample(n=200, random_state=42)
pilot.to_csv("./data/nedostajuci_sinsetovi.csv", index=False)


print("\nSnimljeno:")
print("  - nedostajuci_sinsetovi.csv")

 


Snimljeno:
  - nedostajuci_sinsetovi.csv


In [38]:
len(wordnet) - (duplicates>1).sum()

np.int64(2292)

In [39]:
pilot.shape

(200, 12)

In [40]:
# Matched IDs, synset which has both english and serbian translation, it will be used for verify tranlsation quality
already_translated_synsets = wordnet[wordnet["ID"].isin(srpwn["ID"])]
already_translated_synsets.to_csv("./data/SrpWN_translated_synsets.csv", index = False)

print("\nSnimljeno:")
print("  - SrpWN_translated_synsets.csv")


Snimljeno:
  - SrpWN_translated_synsets.csv
